MIMIC-IV: harmonised exposure definition (calculated SBE), for comparison with eICU-CRD
Runs the data and imputation part of `analysis/abe_aki_analysis.py` (sections 1-3), then fits the primary
model with ΔABE recalculated as in eICU: daily mean standard base excess calculated from pH and bicarbonate
(Van Slyke; bicarbonate from pH and pCO2 by Henderson-Hasselbalch) plus daily mean lactate. Prints aggregate results only.

In [ ]:
# Cell A: setup (clone the code repository, install packages)
import os
if not os.path.exists("/content/abe-aki-mimic-iv"):
    !git clone -q https://github.com/ozkulycolak/abe-aki-mimic-iv.git /content/abe-aki-mimic-iv
%cd /content/abe-aki-mimic-iv/abe-aki-mimic-iv
!pip install -q tableone lifelines
os.environ["ABE_PROJECT_ID"] = "YOUR_PROJECT_ID"   # <- your Google Cloud project
os.environ["ABE_N_IMP"] = "40"


In [ ]:
# Cell B: sections 1-3 of the published analysis script (data, descriptives, multiple imputation)
src = open("analysis/abe_aki_analysis.py").read()
cut = src.index("# 4. PRIMARY ANALYSIS")
exec(src[:cut], globals())
print("Cohort:", len(d0), "| imputed datasets:", len(imps))


In [ ]:
# Cell C: harmonised exposure, SBE from pH and pCO2 (Henderson-Hasselbalch HCO3) + lactate, daily means
SQL_H = f"""
WITH k AS (SELECT stay_id, hadm_id, intime FROM `{PROJECT_ID}.abe.kohort`),
g AS (
  SELECT hadm_id, specimen_id, MIN(charttime) AS charttime,
         MAX(IF(itemid = 50820, valuenum, NULL)) AS ph,
         MAX(IF(itemid = 50818, valuenum, NULL)) AS pco2,
         MAX(IF(itemid = 50802, valuenum, NULL)) AS be
  FROM `physionet-data.mimiciv_3_1_hosp.labevents`
  WHERE itemid IN (50820, 50818, 50802) AND hadm_id IN (SELECT hadm_id FROM k)
  GROUP BY 1, 2
),
gc AS (
  SELECT hadm_id, charttime, ph, be, 0.0307 * pco2 * POW(10, ph - 6.1) AS hco3
  FROM g
  WHERE ph BETWEEN 6.5 AND 8.0 AND pco2 BETWEEN 5 AND 200
),
gp AS (
  SELECT k.stay_id, DATETIME_DIFF(gc.charttime, k.intime, MINUTE) AS t,
         0.9287 * (gc.hco3 - 24.4 + 14.83 * (gc.ph - 7.4)) AS sbe, gc.be
  FROM k JOIN gc USING (hadm_id)
  WHERE gc.hco3 BETWEEN 2 AND 60
    AND gc.charttime BETWEEN DATETIME_SUB(k.intime, INTERVAL 6 HOUR) AND DATETIME_ADD(k.intime, INTERVAL 48 HOUR)
),
lac AS (
  SELECT k.stay_id, DATETIME_DIFF(l.charttime, k.intime, MINUTE) AS t, l.valuenum AS v
  FROM k JOIN `physionet-data.mimiciv_3_1_hosp.labevents` l USING (hadm_id)
  WHERE l.itemid = 50813 AND l.valuenum > 0 AND l.valuenum < 30
    AND l.charttime BETWEEN DATETIME_SUB(k.intime, INTERVAL 6 HOUR) AND DATETIME_ADD(k.intime, INTERVAL 48 HOUR)
),
s AS (
  SELECT stay_id, AVG(IF(t <= 1440, sbe, NULL)) AS sbe_h1, AVG(IF(t > 1440, sbe, NULL)) AS sbe_h2
  FROM gp GROUP BY 1
),
l2 AS (
  SELECT stay_id, AVG(IF(t <= 1440, v, NULL)) AS lak_h1, AVG(IF(t > 1440, v, NULL)) AS lak_h2
  FROM lac GROUP BY 1
)
SELECT k.stay_id, s.sbe_h1, s.sbe_h2, l2.lak_h1, l2.lak_h2
FROM k LEFT JOIN s USING (stay_id) LEFT JOIN l2 USING (stay_id)
"""
SQL_AGREE = f"""
WITH k AS (SELECT hadm_id FROM `{PROJECT_ID}.abe.kohort`),
g AS (
  SELECT specimen_id,
         MAX(IF(itemid = 50820, valuenum, NULL)) AS ph,
         MAX(IF(itemid = 50818, valuenum, NULL)) AS pco2,
         MAX(IF(itemid = 50802, valuenum, NULL)) AS be
  FROM `physionet-data.mimiciv_3_1_hosp.labevents`
  WHERE itemid IN (50820, 50818, 50802) AND hadm_id IN (SELECT hadm_id FROM k)
  GROUP BY 1
),
d AS (
  SELECT 0.9287 * (0.0307 * pco2 * POW(10, ph - 6.1) - 24.4 + 14.83 * (ph - 7.4)) AS sbe, be
  FROM g
  WHERE ph BETWEEN 6.5 AND 8.0 AND pco2 BETWEEN 5 AND 200 AND be BETWEEN -40 AND 40
)
SELECT COUNT(*) AS n, CORR(sbe, be) AS r, AVG(sbe - be) AS mean_diff,
       APPROX_QUANTILES(sbe - be, 4) AS diff_quartiles
FROM d
"""
h = q(SQL_H)
H("H1. HARMONISED EXPOSURE (calculated SBE from pH and pCO2, daily means)")
ag = q(SQL_AGREE)
P(f"Reported BE vs calculated SBE, same specimen: n = {int(ag.n[0])}, r = {ag.r[0]:.3f}, "
  f"mean difference (SBE - BE) = {ag.mean_diff[0]:.2f}, quartiles = {np.round(ag.diff_quartiles[0], 2).tolist()}")

hh = d0[["stay_id"]].merge(h, on="stay_id", how="left")
assert (hh.stay_id.values == d0.stay_id.values).all()
d0["abe_h1"] = (hh.sbe_h1 + hh.lak_h1).values
d0["abe_h2"] = (hh.sbe_h2 + hh.lak_h2).values
d0["delta_h"] = d0.abe_h2 - d0.abe_h1
d0["delta_sbe_h"] = (hh.sbe_h2 - hh.sbe_h1).values
d0["sbe_h1"] = hh.sbe_h1.values
for t in imps:
    for c in ["abe_h1", "abe_h2", "delta_h", "delta_sbe_h", "sbe_h1"]:
        t[c] = d0[c].values

okh = d0.delta_h.notna()
P(f"Harmonised ΔABE available: {int(okh.sum())}/{len(d0)} ({okh.mean()*100:.1f}%); events {int(d0.loc[okh, 'y'].sum())}")
P("ΔABE, published definition, median (IQR):", med_iqr(d0.loc[okh, "delta_abe"], 2),
  "| harmonised:", med_iqr(d0.loc[okh, "delta_h"], 2),
  f"| correlation r = {d0.loc[okh, ['delta_abe', 'delta_h']].corr().iloc[0, 1]:.3f}")
P("Day-1 ABE, published definition:", med_iqr(d0.loc[okh, "abe_ort_0_24"], 2), "| harmonised:", med_iqr(d0.loc[okh, "abe_h1"], 2))

sub = lambda t: t["delta_h"].notna()
sub_s1 = lambda t: t["delta_h"].notna() & (t["evre1_48s"] == 0)
F_ORIG = f"y ~ abe_ort_0_24 + delta_abe + {COV}"
F_H = f"y ~ abe_h1 + delta_h + {COV}"
res = {}
res["orig_same_subset"] = pool(F_ORIG, ["delta_abe"], subset=sub)["delta_abe"]
res["full"] = pool(F_H, ["delta_h"], subset=sub)["delta_h"]
res["creat"] = pool(F_H + " + log_kre_oran_48", ["delta_h"], subset=sub)["delta_h"]
res["nostage1"] = pool(F_H, ["delta_h"], subset=sub_s1)["delta_h"]
res["stage3"] = pool(F_H.replace("y ~", "y_evre3 ~", 1), ["delta_h"], subset=sub)["delta_h"]
res["pars"] = pool(f"y ~ abe_h1 + delta_h + {PARS}", ["delta_h"], subset=sub)["delta_h"]
res["dsbe"] = pool(f"y ~ sbe_h1 + delta_sbe_h + {COV}", ["delta_sbe_h"], subset=sub)["delta_sbe_h"]
P("\nOR per 1 mmol/L increase (model-based SE, Rubin's rules, 40 imputations)")
P("Published definition, same patients:      ", fmt(*res["orig_same_subset"]))
P("Harmonised, full model:                   ", fmt(*res["full"]))
P("Harmonised, + 48-h creatinine change:     ", fmt(*res["creat"]))
P("Harmonised, no KDIGO stage 1 by 48 h:     ", fmt(*res["nostage1"]),
  f"(n = {int((okh & (d0.evre1_48s == 0)).sum())}, events = {int(d0.loc[okh & (d0.evre1_48s == 0), 'y'].sum())})")
P("Harmonised, KDIGO stage 3 or KRT:         ", fmt(*res["stage3"]))
P("Harmonised, parsimonious model:           ", fmt(*res["pars"]))
P("Harmonised ΔSBE (with day-1 SBE):         ", fmt(*res["dsbe"]))


In [ ]:
# Cell D: fixed-effect pooling with eICU-CRD (eICU estimates from the eICU notebook)
from scipy.stats import chi2
EICU = {"full": (0.964, 0.935, 0.994), "creat": (0.976, 0.948, 1.005),
        "nostage1": (0.980, 0.929, 1.034), "stage3": (0.997, 0.966, 1.030)}
H("H2. MIMIC-IV (harmonised) + eICU-CRD, inverse-variance fixed-effect pooling")
for k_, (o, lo, hi) in EICU.items():
    b1, s1 = res[k_][0], res[k_][1]
    b2, s2 = np.log(o), (np.log(hi) - np.log(lo)) / 3.92
    w1, w2 = 1 / s1 ** 2, 1 / s2 ** 2
    bp = (b1 * w1 + b2 * w2) / (w1 + w2); sp = (w1 + w2) ** -0.5
    Q = w1 * (b1 - bp) ** 2 + w2 * (b2 - bp) ** 2
    I2 = max(0, (Q - 1) / Q) * 100 if Q > 0 else 0
    P(f"{k_:9s} MIMIC {np.exp(b1):.3f} | eICU {o:.3f} | pooled {np.exp(bp):.3f} "
      f"({np.exp(bp-1.96*sp):.3f}-{np.exp(bp+1.96*sp):.3f}), P = {pfmt(2*norm.sf(abs(bp/sp)))}; "
      f"heterogeneity P = {pfmt(chi2.sf(Q, 1))}, I2 = {I2:.0f}%")
P("\nDone.")
